# Step 13.5 — Concatenate Heads and Output Projection：怎样把 `(B,H,T,D)` 重新变回 `(B,T,C)`？

Step 13.4 已经把每个 head 的 attention computation 组织成：

$$
O_{heads}
\in
\mathbb R^{B\times H\times T\times D}.
$$

这里每个：

$$
O_{heads}[b,h,t,:]\in\mathbb R^D
$$

都是 sample `b`、head `h`、token position `t` 的 head-specific output。

但是 Transformer 的主 representation width 仍然是：

$$
C=H\times D.
$$

所以 Multi-Head Attention 最后还要做两件事：

```text
1. 把 H 个 head outputs 按 channel 拼回 model width C
2. 用 learned output projection / c_proj 混合这些 head channels
```

本节完成这两步即停，不进入 Step 14.1 的 residual connection。


## 1. 第一步不是直接 Reshape：先把 T / H Axes 换回来

当前：

$$
O_{BHTD}:(B,H,T,D).
$$

我们最终想让每个 token position 的所有 heads 相邻放在一起，所以先交换 H 与 T：

$$
\boxed{
O_{BTHD}
=
\operatorname{swapaxes}(O_{BHTD},1,2)
}
$$

得到：

$$
O_{BTHD}:(B,T,H,D).
$$

精确索引关系：

$$
\boxed{
O_{BTHD}[b,t,h,d]
=
O_{BHTD}[b,h,t,d]
}
$$

这一步仍然只是 axis reordering，不新增参数，也不混合 head 内容。


## 2. Concatenate Heads：把 `(H,D)` 合并回 `C`

因为：

$$
C=H\times D,
$$

所以：

$$
(B,T,H,D)
\rightarrow
\boxed{(B,T,C)}.
$$

用索引写得最清楚：

$$
\boxed{
O_{cat}[b,t,hD+d]
=
O_{BTHD}[b,t,h,d]
}
$$

这就是所谓 **concatenate heads（拼接各头输出）**。

注意 concatenation 不是：

- 把 heads 相加；
- 把 heads 求平均；
- 再做一次 attention；
- 学习新的 parameters。

它只是把：

```text
head 0 的 D channels
head 1 的 D channels
...
head H-1 的 D channels
```

按 feature axis 首尾相接，重新形成长度 C 的 token representation。


## 3. Tiny Example：`H=2, D=2, C=4`

设：

```text
B = 1
T = 2
H = 2
D = 2
C = 4
```

Head 0 对两个 tokens 的 outputs：

$$
O^{(0)}
=
\begin{bmatrix}
1&2\\
3&4
\end{bmatrix}.
$$

Head 1：

$$
O^{(1)}
=
\begin{bmatrix}
10&20\\
30&40
\end{bmatrix}.
$$

因此 `(B,H,T,D)` 中，逻辑结构是：

```text
head 0:
  token 0 -> [1,2]
  token 1 -> [3,4]

head 1:
  token 0 -> [10,20]
  token 1 -> [30,40]
```

交换 H/T 后，每个 token 的 heads 被放到一起：

```text
token 0:
  head 0 -> [1,2]
  head 1 -> [10,20]

token 1:
  head 0 -> [3,4]
  head 1 -> [30,40]
```

再合并 `(H,D)`：

$$
\boxed{
O_{cat}
=
\begin{bmatrix}
1&2&10&20\\
3&4&30&40
\end{bmatrix}
}.
$$

所以 Multi-Head 的“concatenate”本质就是恢复一个长度为 C 的 feature vector。


In [ ]:
import mlx.core as mx

B, H, T, D = 1, 2, 2, 2
C = H * D

# (B,H,T,D)
head_out = mx.array([[
    [
        [1.0, 2.0],
        [3.0, 4.0],
    ],
    [
        [10.0, 20.0],
        [30.0, 40.0],
    ],
]])

# 1) Bring token axis back before head axis: (B,H,T,D) -> (B,T,H,D)
token_major = mx.swapaxes(head_out, 1, 2)

# 2) Merge H and D back into C: (B,T,H,D) -> (B,T,C)
concatenated = mx.reshape(token_major, (B, T, C))

print('head_out:', head_out.shape)
print('token_major:', token_major.shape)
print('concatenated:', concatenated.shape)
print(concatenated)

# Expected when executed:
# [[[ 1,  2, 10, 20],
#   [ 3,  4, 30, 40]]]
# This master notebook remains unexecuted.


## 4. 为什么 Concatenate 完还需要 `c_proj`？

Concatenation 只做 layout recovery：

$$
(B,T,H,D)
\rightarrow
(B,T,C).
$$

它没有让不同 heads 的 channels 发生 learned interaction。

如果直接把 concatenated vector 看成：

$$
o_{cat,t}
=
[
o_t^{(0)}
\mid
o_t^{(1)}
\mid
\cdots
\mid
o_t^{(H-1)}
],
$$

那么不同 head blocks 只是并排放在一起。

Transformer 通常再使用一个 learned **output projection**：

$$
\boxed{
Y
=
O_{cat}W_O
}
$$

其中：

$$
W_O\in\mathbb R^{C\times C}.
$$

在 GPT-style 命名里，这个 projection 常被叫做：

```text
c_proj
```

它的作用是：

> **让输出的每个 C-dimensional feature 都可以学习性地组合来自不同 heads 的 channels。**


## 5. 一个手算 Output Projection：真正混合两个 Heads

沿用：

$$
o_{cat}
=
[1,2,10,20].
$$

这个向量前两个 channels 来自 Head 0，后两个来自 Head 1。

构造一个教学用：

$$
W_O
=
\begin{bmatrix}
1&0&1&0\\
0&1&0&1\\
1&0&-1&0\\
0&1&0&-1
\end{bmatrix}.
$$

按 row-vector convention：

$$
y=o_{cat}W_O.
$$

第一维：

$$
1+10=11.
$$

第二维：

$$
2+20=22.
$$

第三维：

$$
1-10=-9.
$$

第四维：

$$
2-20=-18.
$$

所以：

$$
\boxed{
y=[11,22,-9,-18]
}.
$$

这里可以清楚看到：output projection 的单个输出 feature 可以同时依赖 Head 0 与 Head 1 的 channels。

这个 W_O 只是人为设计的可手算例子；真实 `c_proj` 参数由训练学习。


In [ ]:
W_O = mx.array([
    [1.0, 0.0,  1.0, 0.0],
    [0.0, 1.0,  0.0, 1.0],
    [1.0, 0.0, -1.0, 0.0],
    [0.0, 1.0,  0.0,-1.0],
])  # mathematical row-vector convention: (C,C)

manual_projected = concatenated @ W_O

print('manual projected shape:', manual_projected.shape)
print(manual_projected)

# Expected first token: [11, 22, -9, -18]
# Expected second token: [33, 44, -27, -36]
# This is a hand-designed projection, not learned runtime output.


## 6. MLX 中的 `c_proj = Linear(C,C)`

项目固定 MLX 0.32.2。

可以写：

```python
c_proj = nn.Linear(
    input_dims=C,
    output_dims=C,
    bias=False,
)
```

输入：

$$
O_{cat}:(B,T,C)
$$

输出：

$$
\boxed{
Y:(B,T,C)
}.
$$

所以 output projection **保持 model width C 不变**。

MLX stored weight shape 仍按其 convention 是：

$$
(C,C).
$$

对于方阵这里转置后 shape 仍然是 `(C,C)`，但数学 forward 仍应理解为 row-vector input 乘一个 `C×C` learned linear map。

本节为了把手算与真实 API 分开，不预先声称随机初始化 `c_proj` 的具体输出数值。


In [ ]:
import mlx.nn as nn

c_proj = nn.Linear(
    input_dims=C,
    output_dims=C,
    bias=False,
)

projected = c_proj(concatenated)

print('concatenated:', concatenated.shape)
print('c_proj stored weight:', c_proj.weight.shape)
print('projected:', projected.shape)

# Exact projected values depend on the initialized learned c_proj weights.
# This master notebook remains unexecuted.


## 7. `c_proj` 会增加参数吗？会

Step 13.2 的 fused QKV projection，bias=False 时有：

$$
3C^2
$$

个 weights。

现在 output projection：

$$
W_O:(C,C)
$$

再增加：

$$
C^2
$$

个 weights。

所以只看 Multi-Head Attention 的 QKV + output projections，在 bias=False 情况下：

$$
\boxed{
3C^2+C^2
=
4C^2
}.
$$

如果 QKV fused projection 与 output projection 都使用 bias=True，则 biases 总计：

$$
3C+C=4C.
$$

因此对应总参数为：

$$
\boxed{4C^2+4C}.
$$

这里的 `c_proj` 是新的 learned parameter matrix；这和前面的 transpose / reshape 完全不同。


## 8. 到这里，Multi-Head Attention 的 Shape Surgery 已经闭环

从 Step 13.2 起：

```text
X
(B,T,C)

↓ fused QKV projection

QKV
(B,T,3C)

↓ split

Q/K/V
(B,T,C)

↓ C = H×D, reshape

Q/K/V
(B,T,H,D)

↓ swap T/H

Q/K/V
(B,H,T,D)

↓ head-wise attention

head outputs
(B,H,T,D)

↓ swap H/T back

(B,T,H,D)

↓ merge H×D = C

concatenated
(B,T,C)

↓ c_proj / output projection

Y
(B,T,C)
```

这就是 GPT-style Multi-Head Attention 最核心的 tensor organization。

但这里仍然只是 attention module 的输出；如何把它放进更大的 Transformer repeated block，不属于 Step 13。


## 9. Step 13.5 材料边界

- head-wise output 从 `(B,H,T,D)` 先 `swapaxes(1,2)` 回到 `(B,T,H,D)`，再把 `(H,D)` 合并成 `C=H×D`，得到 `(B,T,C)`。
- concatenate heads 只是 feature-axis layout merge，不是 head 求和、平均，也不是 learned operation。
- 精确索引关系：`O_cat[b,t,hD+d] = O_heads[b,h,t,d]`。
- `c_proj / W_O` 是 learned `C->C` output projection，负责让输出 features 混合来自不同 head blocks 的 channels。
- `c_proj` 保持 shape `(B,T,C)->(B,T,C)`，但增加 `C^2` weights（bias=False）；与 fused QKV 合计为 `4C^2` weights。
- 手工 W_O 例子只是展示跨-head mixing；真实 `c_proj` 权重由训练学习，未执行母版不声称其具体输出。
- 本轮只准备 Step 13.5；不进入 Step 14.1，不讲 residual connection、normalization、MLP 或完整 Transformer block。
